In [1]:
#All the imports
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
import os
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [2]:
def compute_radial_spectrum(fld,dx,dy,dtype=float,mask=None):
    """
    Compute the radial spectra of a 2D field
    input:
    fld : np.array or any 2D field
    dx : (float)
         grid x resolution
    dy : (float)
         grid y resolution
    """
    (nx,ny) = np.shape(np.asarray(fld))
    x = np.arange(dtype(nx)) * dx
    y = np.arange(dtype(ny)) * dy
    kx = np.fft.fftfreq(nx,d=dx) *2*np.pi 
    ky = np.fft.fftfreq(ny,d=dy) *2*np.pi 
    
    #isotropizing
    aa = np.sqrt(np.repeat(kx[np.newaxis,:]**2,ny,axis=0).transpose() + \
                 np.repeat(ky[np.newaxis,:]**2,nx,axis=0))   
    
    #resolution in k_r
    dk = kx[2] - kx[1]
    nk = np.int64(np.max(aa)/dk  + 1)
    kk = np.arange(dtype(nk)) * dk
    i1 = (aa/dk).astype(np.int64)
    i2 = i1 + 1
    s2 = aa/dk - i1.astype(dtype)
    s1 = 1 - s2
    tags=['kk','sp']
    
 
    sps= np.recarray((nk,),dtype=[(i,dtype) for i in tags])
    sps.kk=kk
    sp = np.zeros(nk,dtype=dtype)
    ff=np.abs(np.fft.ifft2(fld))**2
    #if mask is not None:  
    #    if (np.shape(mask) == ff.shape): 
    #    ff * = np.asarray(mask)
    f1 = ff*s1
    f2 = ff*s2

    for i in range(nx):
        for j in range(ny):
            if i2[i,j] < nk :
                sp[i2[i,j]] += f2[i,j] 
                sp[i1[i,j]] += f1[i,j] 
    sps['sp']=sp/dk
    
    
    return sps

In [14]:
base_root = "/home/UNN/w24021992/NAS/simulations_beta_scan"


# base_dirs = [ f"{base_root}/beta_p_0.0625/beta_e_0.25",
#     f"{base_root}/beta_p_0.0625/beta_e_1",
#     f"{base_root}/beta_p_0.0625/beta_e_4",

#     f"{base_root}/beta_p_0.25/beta_e_1",
#     f"{base_root}/beta_p_0.25/beta_e_4",
#     f"{base_root}/beta_p_0.25/beta_e_16",

#     f"{base_root}/beta_p_1/beta_e_0.0625",
#     f"{base_root}/beta_p_1/beta_e_0.25",
#     f"{base_root}/beta_p_1/beta_e_1",
#     f"{base_root}/beta_p_1/beta_e_4",
#     f"{base_root}/beta_p_1/beta_e_16",

#     f"{base_root}/beta_p_4/beta_e_0.0625",
#     f"{base_root}/beta_p_4/beta_e_0.25",
#     f"{base_root}/beta_p_4/beta_e_1",
#     f"{base_root}/beta_p_4/beta_e_4",

    
#     f"{base_root}/beta_p_4/beta_e_16",

#     f"{base_root}/beta_p_16/beta_e_0.0625",
#     f"{base_root}/beta_p_16/beta_e_0.25",
#     f"{base_root}/beta_p_16/beta_e_16",
# ]

base_dirs = [
    
    f"{base_root}/beta_p_16/beta_e_16"
]

In [5]:
dx = 0.0625
dy = 0.0625
kdi = 25

threshold_main = 3.0

for base_dir in base_dirs:
    print(f"Processing: {base_dir}")

    # --- File paths ---
    File_Dn1 = os.path.join(base_dir, "Dn1_ApJ_t1.h5")
    File_Dn2 = os.path.join(base_dir, "Dn1_ApJ_t200.h5")

    # --- Output directory ---
    plot_dir = os.path.join(base_dir, "plots")
    os.makedirs(plot_dir, exist_ok=True)

    # --- Read data ---
    with h5py.File(File_Dn1, 'r') as f:
        data_Dn1 = f['DS1'][:]

    with h5py.File(File_Dn2, 'r') as f:
        data_Dn2 = f['DS1'][:]

    # --- Compute spectra ---
    spectrum_Dn1 = compute_radial_spectrum(data_Dn1, dx, dy)
    spectrum_Dn2 = compute_radial_spectrum(data_Dn2, dx, dy)

    # =========================
    # Density spectrum plot
    # =========================
    plt.figure(figsize=(7,5))
    plt.loglog(spectrum_Dn1.kk, spectrum_Dn1.sp, label=r'$P_n(t=1)$')
    plt.loglog(spectrum_Dn2.kk, spectrum_Dn2.sp, label=r'$P_n(t=200)$')

    plt.xlabel(r'$k$')
    plt.ylabel(r'$P(k)$')
    plt.xlim(5e-2, 50)
    plt.ylim(1e-6, 1e-2)
    plt.legend()
    plt.tight_layout()

    density_plot_path = os.path.join(plot_dir, "Density_Spectrum.png")
    plt.savefig(density_plot_path, dpi=200)
    plt.close()

    # =========================
    # SNR plot
    # =========================
    k = spectrum_Dn1.kk
    snr = spectrum_Dn2.sp / spectrum_Dn1.sp

    idx = np.argmin(np.abs(k - kdi))
    snr_kdi = snr[idx]

    plt.figure(figsize=(7,5))
    plt.loglog(k, snr)

    # Threshold lines
    plt.axhline(threshold_main, linestyle='--', linewidth=1.2,
                color='gray', label='SNR = 3')

    # Grey vertical shading where SNR < 3
    below = snr < threshold_main
    start = None
    for i, val in enumerate(below):
        if val and start is None:
            start = i
        elif not val and start is not None:
            plt.axvspan(k[start], k[i-1], color='lightgray', alpha=0.5)
            start = None
    if start is not None:
        plt.axvspan(k[start], k[-1], color='lightgray', alpha=0.5)

    # Legend additions
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

    noise_patch = Patch(facecolor='lightgray', alpha=0.5,
                        label='Noise-dominated (SNR < 3)')
    red_dot = Line2D([0], [0], marker='o', color='red',
                     linestyle='None',
                     label=r'$SNR\ at\ kd_i=25$')

    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles=handles + [noise_patch, red_dot],
               loc='upper right')

    plt.xlabel(r'$k d_i$')
    plt.ylabel(r'$\mathrm{SNR}(k d_i)$')
    plt.xlim(5e-2, 50)
    plt.tight_layout()

    snr_plot_path = os.path.join(plot_dir, "SNR.png")
    plt.savefig(snr_plot_path, dpi=200)
    plt.close()

    print(f"Saved:")
    print(f"  {density_plot_path}")
    print(f"  {snr_plot_path}\n")


Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
Saved:
  /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/Density_Spectrum.png
  /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/SNR.png



In [15]:
dx = 0.0625
dy = 0.0625

threshold_main = 3.0   # SNR threshold

for base_dir in base_dirs:
    print(f"Processing: {base_dir}")

    # --- File paths ---
    File_Dn1 = os.path.join(base_dir, "Dn1_ApJ_t1.h5")
    File_Dn2 = os.path.join(base_dir, "Dn1_ApJ_t200.h5")

    # --- Read data ---
    with h5py.File(File_Dn1, 'r') as f:
        data_Dn1 = f['DS1'][:]

    with h5py.File(File_Dn2, 'r') as f:
        data_Dn2 = f['DS1'][:]

    # --- Spectra ---
    spectrum_Dn1 = compute_radial_spectrum(data_Dn1, dx, dy)
    spectrum_Dn2 = compute_radial_spectrum(data_Dn2, dx, dy)

    k = spectrum_Dn1.kk
    snr = spectrum_Dn2.sp / spectrum_Dn1.sp

    # ==================================================
    # Find kdi where SNR crosses threshold = 3
    # ==================================================
    crossing_index = None

    for i in range(1, len(k)):
        if snr[i-1] >= threshold_main and snr[i] < threshold_main:
            crossing_index = i
            break

    if crossing_index is None:
        print("SNR never drops below 3.")
        continue

    # ---- Linear interpolation for accurate kdi ----
    k1, k2 = k[crossing_index-1], k[crossing_index]
    s1, s2 = snr[crossing_index-1], snr[crossing_index]

    kdi_snr3 = k1 + (threshold_main - s1) * (k2 - k1) / (s2 - s1)

    print(f"kdi where SNR = 3  -->  {kdi_snr3:.4f}\n")

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_16/beta_e_16
kdi where SNR = 3  -->  0.2311



In [11]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ============================================================
# Parameters
# ============================================================
dx = 0.0625
dy = 0.0625
threshold_main = 3.0

# base_dirs must exist
# compute_radial_spectrum() must exist

for base_dir in base_dirs:

    print(f"Processing: {base_dir}")

    # ------------------------------------------------------------
    # File paths
    # ------------------------------------------------------------
    File_Dn1 = os.path.join(base_dir, "Dn1_ApJ_t0.h5")
    File_Dn2 = os.path.join(base_dir, "Dn1_ApJ_t200.h5")

    plot_dir = os.path.join(base_dir, "plots")
    os.makedirs(plot_dir, exist_ok=True)

    # ------------------------------------------------------------
    # Read data
    # ------------------------------------------------------------
    with h5py.File(File_Dn1, 'r') as f:
        data_Dn1 = f['DS1'][:]

    with h5py.File(File_Dn2, 'r') as f:
        data_Dn2 = f['DS1'][:]

    # ------------------------------------------------------------
    # Compute spectra
    # ------------------------------------------------------------
    spectrum_Dn1 = compute_radial_spectrum(data_Dn1, dx, dy)
    spectrum_Dn2 = compute_radial_spectrum(data_Dn2, dx, dy)

    # ============================================================
    # Density Spectrum
    # ============================================================
    plt.figure(figsize=(7,5))

    plt.loglog(
        spectrum_Dn1.kk,
        spectrum_Dn1.sp,
        label=r'$P_n(t=1)$'
    )

    plt.loglog(
        spectrum_Dn2.kk,
        spectrum_Dn2.sp,
        label=r'$P_n(t=200)$'
    )

    plt.xlabel(r'$k$')
    plt.ylabel(r'$P(k)$')

    plt.xlim(5e-2, 50)
    plt.ylim(1e-6, 1e-2)

    plt.legend()
    plt.tight_layout()

    density_plot_path = os.path.join(plot_dir, "Density_Spectrum.png")
    plt.savefig(density_plot_path, dpi=200)
    plt.close()

    # ============================================================
    # SNR Plot
    # ============================================================
    k = spectrum_Dn1.kk
    snr = spectrum_Dn2.sp / spectrum_Dn1.sp

    fig, ax = plt.subplots(figsize=(7,5))

    ax.loglog(k, snr, zorder=3)

    # ------------------------------------------------------------
    # Threshold line
    # ------------------------------------------------------------
    ax.axhline(
        threshold_main,
        linestyle='--',
        linewidth=1.4,
        color='gray',
        label='SNR = 3',
        zorder=2
    )

    # ------------------------------------------------------------
    # Shade noise region
    # ------------------------------------------------------------
    below = snr < threshold_main
    start = None

    for i, val in enumerate(below):
        if val and start is None:
            start = i
        elif not val and start is not None:
            ax.axvspan(
                k[start],
                k[i-1],
                color='lightgray',
                alpha=0.6,
                zorder=0
            )
            start = None

    if start is not None:
        ax.axvspan(
            k[start],
            k[-1],
            color='lightgray',
            alpha=0.6,
            zorder=0
        )

    # ------------------------------------------------------------
    # Find LEFT noise boundary (SNR >=3 → SNR <3)
    # ------------------------------------------------------------
    cross_idx = None

    for i in range(1, len(k)):
        if snr[i-1] >= threshold_main and snr[i] < threshold_main:
            cross_idx = i
            break

    k_border = None

    if cross_idx is not None:

        k1, k2 = k[cross_idx-1], k[cross_idx]
        s1, s2 = snr[cross_idx-1], snr[cross_idx]

        k_border = k1 + (threshold_main - s1) * (k2 - k1) / (s2 - s1)

        print(f"SNR=3 boundary at kdi = {k_border:.3f}")

        # Vertical boundary line
        ax.axvline(
            k_border,
            color='black',
            linewidth=1.8,
            zorder=5,
            label=r'SNR = 3 boundary'
        )

        # --------------------------------------------------------
        # MARK POSITION ON X-AXIS
        # --------------------------------------------------------
        xticks = list(ax.get_xticks())
        xticks.append(k_border)
        ax.set_xticks(xticks)

        labels = []
        for tick in xticks:
            if np.isclose(tick, k_border):
                labels.append(rf'$k_{{\rm SNR=3}}={k_border:.1f}$')
            else:
                labels.append(None)

        ax.set_xticklabels(labels)

    else:
        print("WARNING: No SNR crossing found.")

    # ------------------------------------------------------------
    # Legend on top
    # ------------------------------------------------------------
    noise_patch = Patch(
        facecolor='lightgray',
        alpha=0.6,
        label='Noise-dominated (SNR < 3)'
    )

    handles, labels = ax.get_legend_handles_labels()

    legend = ax.legend(
        handles=handles + [noise_patch],
        loc='upper right',
        frameon=True
    )

    legend.set_zorder(100)

    # ------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------
    ax.set_xlabel(r'$k d_i$')
    ax.set_ylabel(r'$\mathrm{SNR}(k d_i)$')
    ax.set_xlim(5e-2, 50)

    plt.tight_layout()

    snr_plot_path = os.path.join(plot_dir, "SNR.png")
    plt.savefig(snr_plot_path, dpi=200)
    plt.close()

    # ============================================================
    # Done
    # ============================================================
    print("Saved:")
    print(f"  {density_plot_path}")
    print(f"  {snr_plot_path}\n")

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
SNR=3 boundary at kdi = 27.734
Saved:
  /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/Density_Spectrum.png
  /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/SNR.png

